In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
from torchsummary import summary

In [2]:
# Download training data from open datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [3]:
classes = training_data.classes
print(classes)

num_training_samples = len(training_data)
print(f"Number of samples in the training data: {num_training_samples}")

num_test_samples = len(test_data)
print(f"Number of samples in the test data: {num_test_samples}")

['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']
Number of samples in the training data: 60000
Number of samples in the test data: 10000


In [5]:
batch_size = 64 # we will talk about it later

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

# for X, y in train_dataloader:
#     print(f"Shape of X [N, C, H, W]: {X.shape}")
#     print(f"Shape of y: {y.shape} {y.dtype}")

#     fig, axes = plt.subplots(1, 6, figsize=(10, 10))
#     axes = axes.flatten()

#     for i in range(6):
#         img = X[i].squeeze().numpy()
#         axes[i].imshow(img, cmap="grey")
#         axes[i].set_title(f"Label: {y[i]}")
#         axes[i].axis('off')

#     plt.tight_layout()
#     plt.show()

#     break

In [7]:
device = (
    "cuda" if torch.cuda.is_available()
     else "cpu"
)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, bias=True),   #Size 24x24 
            nn.ReLU(inplace=True),                                                         
            nn.MaxPool2d(kernel_size=2, stride=2),                                          #Size 12x12  
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, bias=True),  #Size 8x8
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                                          #Size 4x4
            
            nn.Flatten(),
            nn.Linear(in_features=4*4*16, out_features=256, bias=True),                     #When flattened: Size 4x4 times 16 channels
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=120, bias=True),
            nn.ReLU(),
            nn.Linear(in_features=120, out_features=84, bias=True),
            nn.ReLU(),
            nn.Linear(in_features=84, out_features=10, bias=True)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits
    
model = CNN().to(device)
summary(model, input_size=(1, 28, 28), device=device)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 6, 24, 24]             156
              ReLU-2            [-1, 6, 24, 24]               0
         MaxPool2d-3            [-1, 6, 12, 12]               0
            Conv2d-4             [-1, 16, 8, 8]           2,416
              ReLU-5             [-1, 16, 8, 8]               0
         MaxPool2d-6             [-1, 16, 4, 4]               0
           Flatten-7                  [-1, 256]               0
            Linear-8                  [-1, 256]          65,792
              ReLU-9                  [-1, 256]               0
           Linear-10                  [-1, 120]          30,840
             ReLU-11                  [-1, 120]               0
           Linear-12                   [-1, 84]          10,164
             ReLU-13                   [-1, 84]               0
           Linear-14                   

In [ ]:
loss_fn = nn.CrossEntropyLoss()

learning_rate = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    total_loss, correct = 0, 0  # Track the total loss for the epoch

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)
        total_loss += loss.item()  # Accumulate loss

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Calculate accuracy
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            
    avg_loss = total_loss / len(dataloader)  # Calculate average loss for the epoch
    avg_accuracy = correct / size

    train_losses.append(avg_loss)  # Store the average loss for the epoch
    train_accuracies.append(avg_accuracy) # Store the average accuracy for the epoch
    return avg_accuracy, avg_loss

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    test_losses.append(test_loss)  # Store the test loss for the epoch
    test_accuracies.append(correct)
    
    return correct, test_loss

In [ ]:
epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_acc, train_loss = train(train_dataloader, model, loss_fn, optimizer)
    test_acc, test_loss = test(test_dataloader, model, loss_fn)
    print(f"Train accuracy: {round(train_acc,4)}, Train avg loss: {round(train_loss,4)}")
    print(f"Test accuracy: {round(test_acc, 4)}, Test avg loss {round(test_loss,4)}")
    print("\n-------------------------------")
    if test_acc > 0.99:
        epochs = t+1
        break
print("Done!")


In [ ]:
plt.subplot(2,1,1)
plt.plot(range(1, epochs+1), train_losses, label="Train Loss")
plt.plot(range(1, epochs+1), test_losses, label="Test Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Testing Loss over Epochs")
plt.legend()
plt.show()

plt.subplot(2,1,2)
plt.plot(range(1, epochs+1), train_accuracies, label="Train Accuracy")
plt.plot(range(1, epochs+1), test_accuracies, label="Test Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training and Testing Accuracy over Epochs")
plt.legend()
plt.show()

In [ ]:
model.eval()
wrong_predicts = []
for idx in range(len(test_data)):
    x, y = test_data[idx][0], test_data[idx][1]
    # plt.imshow(x.permute(1,2,0), cmap = 'gray')
    with torch.no_grad():
        x = x.to(device)
        pred = model(x.unsqueeze(0))
        predicted, actual = classes[pred[0].argmax(0)], classes[y]
        # print(f'Predicted: "{predicted}", Actual: "{actual}"')
    if predicted != actual:
        point = [idx, predicted, actual]
        wrong_predicts.append(point)

#Showing the first 10 wrong predictions of the test dataset
fig, ax = plt.subplots(2,5, figsize=(10,5))
for x in range(2):
    for i in range(5):
        idx, pred, actual = wrong_predicts[i + x*5]
        ax[x][i].imshow(test_data[idx][0].squeeze(0),cmap='gray')
        ax[x][i].set_title(f"Pred: {pred[0]} / Actual: {actual[0]}")
plt.tight_layout()
plt.show()